# Sample EMA VP-SDE checkpoint

Notebook downloads the EMA-only checkpoint from the same Yandex Disk folder as the dataset, loads validation/test fields, samples from the score model, and plots generated and real fields.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/SadreevAmir/DL_final_project.git"
REPO_DIR = "dl_final_project"

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

In [ ]:
DATA_SOURCE = "https://disk.yandex.ru/d/rrjDGzzX5cfFnA"
EMA_CHECKPOINT_NAME = "best_score_vp_kolmogorov_velocity_256_to_64_coords_2ch_64x64_coords_bf16_20260517_221201_epoch_0014_val_0.028523_ema_model_only.pt"

CACHE_DIR = Path("data/inference_cache")
CHECKPOINT_DIR = Path("checkpoints")
STATS_FILENAME = "kolmogorov_velocity_256_to_64_train_stats.json"

VAL_LIMIT = 16
TEST_LIMIT = 16
SAMPLE_COUNT = 8
SAMPLE_STEPS = 256
SEED = 7

CHANNELS = 2
IMAGE_SIZE = 64
COORDINATE_MODE = "fourier"
CHANNELS_PER_LEVEL = (96, 192, 384)
NUM_RES_BLOCKS = 3
ATTENTION_HEAD_DIM = 32
PADDING_MODE = "circular"
TIME_EMBEDDING_SCALE = 999.0
CLIP_PRED_X0 = 5.0

In [ ]:
import hashlib
import json
import math
import urllib.parse
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from score_training import DiffusersUNet, VPCosineSDE
from score_training.data import _download_yandex_public_folder

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [ ]:
def yandex_download_href(public_key: str, path: str) -> str:
    params = {"public_key": public_key, "path": path}
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(api_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        payload = json.loads(response.read().decode("utf-8"))
    return payload["href"]


def download_file(url: str, target: Path, desc: str) -> Path:
    target.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        total = int(response.headers.get("Content-Length", 0))
        if target.exists() and (total <= 0 or target.stat().st_size == total):
            print("using cached:", target)
            return target
        with target.open("wb") as handle, tqdm(total=total or None, unit="B", unit_scale=True, desc=desc) as bar:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                handle.write(chunk)
                bar.update(len(chunk))
    return target


checkpoint_path = CHECKPOINT_DIR / EMA_CHECKPOINT_NAME
href = yandex_download_href(DATA_SOURCE, "/" + EMA_CHECKPOINT_NAME)
download_file(href, checkpoint_path, EMA_CHECKPOINT_NAME)
print("checkpoint:", checkpoint_path, f"{checkpoint_path.stat().st_size / 1024**2:.1f} MB")

In [ ]:
dataset_dir = _download_yandex_public_folder(DATA_SOURCE, CACHE_DIR)
npz_files = sorted(dataset_dir.rglob("*.npz"))
json_files = sorted(dataset_dir.rglob("*.json"))
print("dataset dir:", dataset_dir)
print("npz files:", len(npz_files))
print("json files:", [p.name for p in json_files])

In [ ]:
def split_from_path(path: Path) -> str | None:
    name = path.stem.lower()
    if name == "train" or name.startswith("train_"):
        return "train"
    if name == "val" or name.startswith("val_"):
        return "val"
    if name == "test" or name.startswith("test_"):
        return "test"
    return None


def find_stats_file() -> Path | None:
    preferred = [p for p in json_files if p.name == STATS_FILENAME]
    candidates = preferred + json_files
    for path in candidates:
        try:
            payload = json.loads(path.read_text())
        except Exception:
            continue
        if "mean" in payload and "std" in payload:
            return path
    return None


def compute_train_stats() -> tuple[np.ndarray, np.ndarray]:
    total_count = 0
    sum_channels = None
    sumsq_channels = None
    train_files = [p for p in npz_files if split_from_path(p) == "train"]
    if not train_files:
        raise RuntimeError("No train shards found and no stats JSON is available.")
    for path in tqdm(train_files, desc="computing train stats", unit="file"):
        with np.load(path) as arrays:
            x = arrays["images"].astype(np.float64, copy=False)
        n = x.shape[0] * x.shape[2] * x.shape[3]
        current_sum = x.sum(axis=(0, 2, 3))
        current_sumsq = (x * x).sum(axis=(0, 2, 3))
        sum_channels = current_sum if sum_channels is None else sum_channels + current_sum
        sumsq_channels = current_sumsq if sumsq_channels is None else sumsq_channels + current_sumsq
        total_count += n
    mean = sum_channels / total_count
    var = np.maximum(sumsq_channels / total_count - mean * mean, 1.0e-12)
    return mean.astype(np.float32), np.sqrt(var).astype(np.float32)


stats_file = find_stats_file()
if stats_file is not None:
    stats = json.loads(stats_file.read_text())
    mean = np.asarray(stats["mean"], dtype=np.float32)
    std = np.asarray(stats["std"], dtype=np.float32)
    print("stats loaded from:", stats_file)
else:
    mean, std = compute_train_stats()
    print("stats computed from train shards")

std = np.maximum(std, 1.0e-6)
print("mean:", mean)
print("std:", std)

In [ ]:
def load_split(split_name: str, limit: int | None = None) -> np.ndarray:
    parts = []
    loaded = 0
    files = [p for p in npz_files if split_from_path(p) == split_name]
    if not files:
        raise RuntimeError(f"No {split_name} shards found in {dataset_dir}")
    for path in files:
        with np.load(path) as arrays:
            x = arrays["images"].astype(np.float32, copy=False)
        if limit is not None:
            remaining = limit - loaded
            if remaining <= 0:
                break
            x = x[:remaining]
        parts.append(x)
        loaded += x.shape[0]
    return np.ascontiguousarray(np.concatenate(parts, axis=0), dtype=np.float32)


val_raw = load_split("val", VAL_LIMIT)
test_raw = load_split("test", TEST_LIMIT)
val_norm = (val_raw - mean.reshape(1, -1, 1, 1)) / std.reshape(1, -1, 1, 1)
test_norm = (test_raw - mean.reshape(1, -1, 1, 1)) / std.reshape(1, -1, 1, 1)
print("val raw:", val_raw.shape, "val norm:", val_norm.shape)
print("test raw:", test_raw.shape, "test norm:", test_norm.shape)

In [ ]:
def make_coord_grid(height: int, width: int, device: torch.device, mode: str = "fourier") -> torch.Tensor:
    if mode == "fourier":
        x_angles = torch.arange(width, device=device, dtype=torch.float32) * (2.0 * math.pi / width)
        y_angles = torch.arange(height, device=device, dtype=torch.float32) * (2.0 * math.pi / height)
        yy, xx = torch.meshgrid(y_angles, x_angles, indexing="ij")
        return torch.stack((torch.sin(xx), torch.cos(xx), torch.sin(yy), torch.cos(yy)), dim=0).unsqueeze(0)
    if mode == "linear":
        y = torch.linspace(-1.0, 1.0, height, device=device)
        x = torch.linspace(-1.0, 1.0, width, device=device)
        yy, xx = torch.meshgrid(y, x, indexing="ij")
        return torch.stack((xx, yy), dim=0).unsqueeze(0)
    raise ValueError(f"Unknown coordinate mode: {mode}")


coords = make_coord_grid(IMAGE_SIZE, IMAGE_SIZE, device, COORDINATE_MODE)
model = DiffusersUNet(
    in_channels=CHANNELS + int(coords.shape[1]),
    out_channels=CHANNELS,
    channels_per_level=CHANNELS_PER_LEVEL,
    num_res_blocks=NUM_RES_BLOCKS,
    image_size=IMAGE_SIZE,
    attention_head_dim=ATTENTION_HEAD_DIM,
    padding_mode=PADDING_MODE,
).to(device)

state = torch.load(checkpoint_path, map_location="cpu")
model.load_state_dict(state)
model.eval()
sde = VPCosineSDE().to(device)
print("loaded EMA tensors:", len(state))

In [ ]:
@torch.no_grad()
def sample_raw(count: int = SAMPLE_COUNT, steps: int = SAMPLE_STEPS) -> np.ndarray:
    shape = (count, CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
    samples_norm = sde.sample(
        model=model,
        shape=shape,
        coords=coords,
        steps=steps,
        device=device,
        time_embedding_scale=TIME_EMBEDDING_SCALE,
        clip_pred_x0=CLIP_PRED_X0,
    )
    x = samples_norm.float().cpu().numpy()
    return x * std.reshape(1, -1, 1, 1) + mean.reshape(1, -1, 1, 1)


samples_raw = sample_raw()
print("samples:", samples_raw.shape)

In [ ]:
def vorticity(fields: np.ndarray) -> np.ndarray:
    ux = fields[:, 0]
    uy = fields[:, 1]
    h, w = ux.shape[-2:]
    kx = (2.0 * np.pi * np.fft.fftfreq(w)).astype(np.float32)
    ky = (2.0 * np.pi * np.fft.fftfreq(h)).astype(np.float32)
    ux_hat = np.fft.fft2(ux)
    uy_hat = np.fft.fft2(uy)
    duy_dx = np.fft.ifft2(1j * kx[None, None, :] * uy_hat).real
    dux_dy = np.fft.ifft2(1j * ky[None, :, None] * ux_hat).real
    return (duy_dx - dux_dy).astype(np.float32)


def robust_absmax(x: np.ndarray, q: float = 99.0) -> float:
    value = float(np.percentile(np.abs(x), q))
    return max(value, 1.0e-6)


def plot_fields(fields: np.ndarray, title: str, count: int = 8) -> None:
    fields = fields[:count]
    omega = vorticity(fields)
    vmax_ux = robust_absmax(fields[:, 0])
    vmax_uy = robust_absmax(fields[:, 1])
    vmax_om = robust_absmax(omega)
    fig, axes = plt.subplots(len(fields), 3, figsize=(7.5, 2.1 * len(fields)))
    if len(fields) == 1:
        axes = axes[None, :]
    axes[0, 0].set_title("u_x")
    axes[0, 1].set_title("u_y")
    axes[0, 2].set_title("vorticity")
    for i in range(len(fields)):
        axes[i, 0].imshow(fields[i, 0], cmap="RdBu_r", vmin=-vmax_ux, vmax=vmax_ux)
        axes[i, 1].imshow(fields[i, 1], cmap="RdBu_r", vmin=-vmax_uy, vmax=vmax_uy)
        axes[i, 2].imshow(omega[i], cmap="RdBu_r", vmin=-vmax_om, vmax=vmax_om)
        for ax in axes[i]:
            ax.axis("off")
    fig.suptitle(title, y=1.0)
    fig.tight_layout()
    plt.show()


plot_fields(samples_raw, "Generated samples", SAMPLE_COUNT)
plot_fields(val_raw, "Validation fields", min(VAL_LIMIT, 8))
plot_fields(test_raw, "Test fields", min(TEST_LIMIT, 8))